# Capstone: Optimización Estratégica de Minería a Cielo Abierto
## Proyecto Final: Certificación Google Data Analytics
**Analista:** Cesar Contreras  
**Yacimiento:** Marvin (Cobre/Oro)

---

## 1. Introducción y Contexto del Proyecto
Este proyecto representa la culminación de la Certificación de Google Data Analytics, aplicado a la **Optimización del Límite Final del Pit (Ultimate Pit Limit - UPIT)** y el **Secuenciamiento de Producción**. El objetivo es transformar un "gemelo digital" geológico en una decisión estratégica que maximice el valor económico, respetando las restricciones físicas de estabilidad y las capacidades operativas de la planta y la mina.

## 2. Fuente de Datos (MineLib)
Los datos provienen de **MineLib**, el repositorio estándar de la industria para problemas de optimización minera.
* **Instancia:** Marvin (Yacimiento polimetálico de Cobre y Oro).
* **Archivos del Proyecto:**
    * `marvin.blocks`: Información de leyes, posición y masa por bloque.
    * `marvin.prec`: Estructura de precedencias que define la geometría de excavación.
    * `marvin.upit / .cpit / .pcpsp`: Formulación de problemas de optimización.

## 3. Fundamentos Técnicos (Voxelización)
El yacimiento se discretiza en bloques tridimensionales conocidos como **Voxels**.
* **Dimensiones del Bloque:** 30x30x30 m
* **Volumen por Bloque:** 27,000 m³
* **Densidad Estimada:** 2.22 t/m³
* **Masa por Bloque:** 60,000 t (Promedio)
* **Geometría de Excavación:** Ángulo de talud de 45° calculado con 8 niveles de precedencia para garantizar la estabilidad de las paredes del rajo.

## 4. Parámetros Económicos y Operativos
Para determinar la rentabilidad de cada unidad de roca, se definen los siguientes parámetros de costos y mercado:

| Categoría | Parámetro | Valor |
| :--- | :--- | :--- |
| **Costos** | Costo de Minado (Mine Cost) | 0.9 $/t |
| | Costo de Procesamiento (Proc Cost) | 4.0 $/t |
| **Metales** | Precio Oro (Price AU) | 12.0 $/g |
| | Costo Comercialización Oro (Selling AU) | 0.2 $/g |
| | Precio Cobre (Price CU) | 20.0 $/% |
| | Costo Comercialización Cobre (Selling CU) | 7.2 $/% |
| **Metalurgia**| Recuperación Oro (Rec AU) | 0.6 |
| | Recuperación Cobre (Rec CU) | 0.88 |
| **Finanzas** | Tasa de Descuento (Discount Rate) | 0.1 |

## 5. Cálculo de Factores Netos de Retorno (NSR)
Calculamos la rentabilidad neta por unidad de ley para cada metal antes de descontar los costos fijos de procesamiento.

### Factor Neto del Oro (Au)
$$Factor_{AU} = (Price_{AU} - Selling_{AU}) \times Rec_{AU}$$
$$Factor_{AU} = (12.0 - 0.2) \times 0.6 = 7.08$$

### Factor Neto del Cobre (Cu)
$$Factor_{CU} = (Price_{CU} - Selling_{CU}) \times Rec_{CU}$$
$$Factor_{CU} = (20.0 - 7.2) \times 0.88 = 11.264$$

## 6. Algoritmo de Valorización Final
La unificación económica de ambos metales define el margen de contribución de cada bloque hacia el beneficio total del proyecto. Se utiliza la siguiente fórmula para el cálculo del beneficio de procesamiento:

$$proc\_profit = (AU \times 7.08) + (CU \times 11.264) - 4.0$$

*Un bloque se enviará a planta (mineral) si y solo si $proc\_profit > 0$.*

## 7. Restricciones de Capacidad y Benchmark
* **Capacidad de Mina (Extracción Total):** 60M t/año
* **Capacidad de Planta (Procesamiento):** 20M t/año
* **Valor Objetivo (Benchmark):** 1,415,655,436 $ (UPIT sin descuento).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Definición de constantes económicas (basadas en nuestra teoría)
FACTOR_AU = 7.08
FACTOR_CU = 11.264
PROC_COST = 4.0

# 2. Ingesta de datos optimizada
# El archivo marvin.blocks contiene: id, x, y, z, tonn, au, cu, proc_profit
columnas = ['id', 'x', 'y', 'z', 'tonn', 'au', 'cu', 'proc_profit']

print("Cargando modelo de bloques Marvin...")
df_blocks = pd.read_csv(
    'marvin/marvin.blocks', 
    sep=r'\s+', 
    skiprows=1, 
    names=columnas,
    dtype={'id': np.int32, 'x': np.float32, 'y': np.float32, 'z': np.float32}
)

# 3. Validación inicial de la estructura (Fase Process)
print(f"Total de bloques cargados: {len(df_blocks)}")

# Verificamos los primeros registros
display(df_blocks.head())

# Cálculo de la masa total del yacimiento para verificar integridad
masa_total_mt = df_blocks['tonn'].sum() / 1e6
print(f"Masa total del yacimiento: {masa_total_mt:.2f} Million Tonnes (Mt)")

Cargando modelo de bloques Marvin...
Total de bloques cargados: 53270


,id,x,y,z,tonn,au,cu,proc_profit
0,1,0.0,0.0,1.0,61200.012,0.0,0.0,-4.0
1,2,0.0,0.0,2.0,61200.012,0.0,0.0,-4.0
2,3,0.0,0.0,3.0,61200.012,0.0,0.0,-4.0
3,4,0.0,0.0,4.0,61200.012,0.0,0.0,-4.0
4,5,0.0,0.0,5.0,61200.012,0.0,0.0,-4.0


Masa total del yacimiento: 3186.56 Million Tonnes (Mt)
